# Transcriptomic Expression Profiles and Gene-Disease-Pathway Relationships in Cardiac Tissue Samples Exploration with `mlcroissant`
This notebook loads and explores the FAIR^2 pilot transcriptomics dataset using the `mlcroissant` library.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.6457-3rhp/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.6457-3rhp/fair2.json'

# Load dataset package
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(metadata.name)
print(metadata.description)


## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

- Each record set, field, and column is identified by its unique `@id`.

In [ ]:
# List all record sets and their fields referenced by @id
record_sets = getattr(metadata, 'recordSet', [])

if not record_sets:
    # If no recordSet is provided in metadata, attempt to enumerate from the schema
    # The Croissant implementation allows dataset.record_sets property
    record_sets = [rs['@id'] for rs in dataset.record_sets]

print('Record sets available:')
for rs in dataset.record_sets:
    print(f"RecordSet @id: {rs['@id']}, name: {rs.get('name','')}")
    print('  Fields:')
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"    Field @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")
    print('  Columns:')
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    for col in columns:
        print(f"    Column @id: {col['@id']}, name: {col.get('name','')}, source: {col.get('source','')}")
print('\n')


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

- Use `@id` to reference record sets.
- Print available columns for a chosen record set.

In [ ]:
# Make a list of all record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
    except Exception as e:
        print(f"Could not load records from {rsid}: {str(e)}")

# Choose the main gene expression matrix record set (most likely FPKM data)
# Find the record set whose name or @id contains 'FPKM' or 'expression'
expr_rs_candidates = [rs for rs in dataset.record_sets if 'FPKM' in rs.get('name','').upper() or 'expression' in rs.get('name','').lower()]
if expr_rs_candidates:
    expr_rs = expr_rs_candidates[0]
    expr_rs_id = expr_rs['@id']
    print(f"Using record set for expression matrix: {expr_rs_id}")
else:
    expr_rs_id = record_set_ids[0] # fallback
    print(f"Fallback to first record set: {expr_rs_id}")

df_expr = dataframes.get(expr_rs_id, pd.DataFrame())
print("Columns in expression record set:", df_expr.columns.tolist())
df_expr.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
 - Filtering records based on criteria (e.g., gene expression levels)
 - Normalizing numeric fields (e.g., log2 or z-score)
 - Grouping by disease status
 
**All operations use field or column `@id` references.**

In [ ]:
# Assume the expression matrix has gene expression values for each sample (columns per sample, rows per gene)

# Identify numeric columns (sample columns) -- usually field @id or column @id in Croissant
numeric_columns = [col for col in df_expr.columns if df_expr[col].dtype in [np.float64, np.int64, float, int]]
if not numeric_columns:
    # Try to guess columns based on common names
    possible_numeric = [col for col in df_expr.columns if 'FPKM' in col.upper() or 'expression' in col.lower() or col.lower().startswith('sample_')]
    numeric_columns = possible_numeric

# Select the first numeric field (by column @id)
if numeric_columns:
    numeric_field_id = numeric_columns[0]
else:
    numeric_field_id = df_expr.columns[0]
print(f"Using numeric field: {numeric_field_id}")

# Set a threshold (arbitrarily 10)
threshold = 10
filtered_df = df_expr[df_expr[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head()])

# Group by disease status field (find a field @id related to disease)
group_field = None
for col in df_expr.columns:
    if 'disease' in col.lower() or 'status' in col.lower():
        group_field = col
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize expression distribution and relationship with disease status using column `@id` references.

In [ ]:
# Plot histogram of chosen numeric field (gene expression)
plt.figure(figsize=(8, 4))
sns.histplot(df_expr[numeric_field_id], bins=40, kde=True)
plt.title(f"Distribution of {numeric_field_id} expression values")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Plot boxplot/grouped bar by disease status (if available)
if group_field:
    plt.figure(figsize=(6, 4))
    sns.boxplot(x=group_field, y=numeric_field_id, data=df_expr)
    plt.title(f"{numeric_field_id} Expression by {group_field}")
    plt.show()

## 6. Conclusion
Summarize the key findings:
- Loaded metadata and data using `mlcroissant` from FAIR^2 Croissant schema URL.
- Inspected record sets and fields via their `@id`s.
- Filtered and normalized gene expression, grouped by disease status.
- Visualized gene expression distribution and compared groups.

This notebook can be adapted for more advanced analyses by referencing additional record set, field, and column `@id`s.